# Portfolio


Strategy: **market_synchronous**.

## Settings

- **Purpose:** Inspect the two independently funded synchronous books.
- **Settings:** Half the initial account per book; five longs and five shorts per book; one-minute cutoffs.
- **Data:** Shared model predictions and exchange sessions.
- **Findings:**
- **Decision:** No transfers between books; duplicates across books are allowed.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.preprocessing.market_data import ResearchPaths, sessions
from src.backtesting.portfolio_management import load_prediction_events
from src.backtesting.portfolio_management import entry_schedule, select_synchronous

PROJECT_ROOT = Path.cwd().resolve().parents[2]
from src.backtesting.portfolio_management import PortfolioSettings
PERIOD = "2025-02-01_2025-12-31"
DATA_START = pd.Timestamp("2025-01-01", tz="UTC")
END = pd.Timestamp("2026-01-01", tz="UTC")
HORIZON_BARS = 1000
K = 5
STEP_SIZE = 0.10
INITIAL_AUM = 100_000.0
BROKER_FEE_BPS = 1.0
SLIPPAGE_BPS = 1.0
HALF_LIFE_MINUTES = 390.0
paths = ResearchPaths(PROJECT_ROOT, model_kind="market", period=PERIOD)
STRATEGY = "market_synchronous"
settings = PortfolioSettings(
    k=K, step_size=STEP_SIZE, initial_aum=INITIAL_AUM,
    broker_fee_bps=BROKER_FEE_BPS, slippage_bps=SLIPPAGE_BPS,
    half_life_minutes=HALF_LIFE_MINUTES,
)
events = load_prediction_events(paths, horizon_bars=HORIZON_BARS)
development = events.loc[events.partition.eq("development")]
calendar = sessions(paths, start=DATA_START, end=END)
end = calendar.loc[calendar.close.lt(development.holdout_boundary.iloc[0]), "close"].max()
schedule = entry_schedule(calendar, end)
selected = select_synchronous(development, schedule, settings.k)
selected = selected.loc[selected.exit.notna() & selected.exit.le(end)]
display(selected[["symbol", "event_start", "group", "entry", "primary_side", "meta_probability", "target_weight"]].head(20))

display(pd.Series(vars(settings), name="value").to_frame())

## Candidates and Weights

- **Purpose:** Inspect latest-symbol predictions ranked by meta take probability.
- **Settings:** Keep meta_action=1; separate primary directions; alphabetical ties.
- **Data:** First eligible development round.
- **Findings:**
- **Decision:** Each signal belongs to one round only; unavailable future features cannot qualify.

In [ ]:
example = selected.loc[selected.entry.eq(selected.entry.min())]
display(example[["group", "symbol", "primary_side", "meta_probability", "target_weight", "entry", "exit"]])

## Scheduled Entry and Exit

- **Purpose:** Define scheduled whole-position round trips.
- **Settings:** Open-to-same-close or close-to-next-close; first/last regular-session tick proxies.
- **Data:** Exchange calendar and raw trades.
- **Findings:**
- **Decision:** Close matured positions before new entries, including full round-trip costs for repeated symbols.

In [ ]:
rules = pd.Series({
    "cutoff": "entry minus one minute; later signals move to the next round",
    "entry_price": "first/last regular-session tick proxy, not an auction guarantee",
    "missing_entry": "skip the order and leave its allocation in cash",
    "missing_exit": "stop evaluation and report the missing price",
    "final_entry": "only if the scheduled exit is within the evaluation period",
    "capital": "independent book PnL; half of pre-cost book equity per direction",
    "reentry": "liquidate fully, then enter again with both costs",
    "valuation": "session boundary snapshots; intraday drawdowns are not measured",
}, name="policy")
display(rules.to_frame())